<a href="https://colab.research.google.com/github/Adijais124/NLP-lab/blob/main/nlp2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import nltk
import pandas as pd

# Download required NLTK datasets
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('brown')
nltk.download('punkt')

from nltk.corpus import wordnet as wn

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [2]:
# List of 10 selected words
words_q1 = ['car', 'bank', 'bat', 'dog', 'light', 'plant', 'book', 'teacher', 'student', 'computer']

q1_records = []

for word in words_q1:
    synsets = wn.synsets(word)
    for syn in synsets:
        # Synset name, POS, definition, and examples
        q1_records.append({
            'Target Word': word,
            'Synset Name': syn.name(),
            'POS': syn.pos(),
            'Definition': syn.definition(),
            'Examples': "; ".join(syn.examples()) if syn.examples() else "N/A"
        })

df_q1 = pd.DataFrame(q1_records)

# Display sample output (first synset per word for concise display)
df_sample = df_q1.groupby('Target Word').head(1).reset_index(drop=True)
print(df_sample[['Target Word', 'Synset Name', 'Definition', 'Examples']])

  Target Word    Synset Name  \
0         car       car.n.01   
1        bank      bank.n.01   
2         bat       bat.n.01   
3         dog       dog.n.01   
4       light     light.n.01   
5       plant     plant.n.01   
6        book      book.n.01   
7     teacher   teacher.n.01   
8     student   student.n.01   
9    computer  computer.n.01   

                                          Definition  \
0  a motor vehicle with four wheels; usually prop...   
1  sloping land (especially the slope beside a bo...   
2  nocturnal mouselike mammal with forelimbs modi...   
3  a member of the genus Canis (probably descende...   
4  (physics) electromagnetic radiation that can p...   
5         buildings for carrying on industrial labor   
6  a written work or composition that has been pu...   
7              a person whose occupation is teaching   
8  a learner who is enrolled in an educational in...   
9  a machine for performing calculations automati...   

                              

In [3]:
# 10 selected noun concepts
words_q2 = ['car', 'bank', 'bat', 'dog', 'tree', 'apple', 'teacher', 'computer', 'chair', 'lake']

def get_hypernym_chain(synset, depth=3):
    """Recursively traces hypernyms up the hierarchy up to a given depth."""
    chain = [synset.name().split('.')[0]]
    current = synset
    for _ in range(depth):
        hypernyms = current.hypernyms()
        if not hypernyms:
            break
        current = hypernyms[0]
        chain.append(current.name().split('.')[0])
    return " -> ".join(chain)

q2_data = []

for word in words_q2:
    syn = wn.synsets(word, pos=wn.NOUN)[0]  # Select primary noun synset
    direct_hypernyms = [h.name().split('.')[0] for h in syn.hypernyms()]
    hierarchy_chain = get_hypernym_chain(syn, depth=3)

    q2_data.append({
        'Word': word,
        'Synset': syn.name(),
        'Direct Hypernym(s)': ", ".join(direct_hypernyms) if direct_hypernyms else "None",
        'Hierarchy Chain (Level 0 -> 1 -> 2 -> 3)': hierarchy_chain
    })

df_q2 = pd.DataFrame(q2_data)
print(df_q2.to_string(index=False))

    Word        Synset      Direct Hypernym(s)                          Hierarchy Chain (Level 0 -> 1 -> 2 -> 3)
     car      car.n.01           motor_vehicle car -> motor_vehicle -> self-propelled_vehicle -> wheeled_vehicle
    bank     bank.n.01                   slope                   bank -> slope -> geological_formation -> object
     bat      bat.n.01               placental                          bat -> placental -> mammal -> vertebrate
     dog      dog.n.01 domestic_animal, canine                      dog -> domestic_animal -> animal -> organism
    tree     tree.n.01             woody_plant                    tree -> woody_plant -> vascular_plant -> plant
   apple    apple.n.01      edible_fruit, pome          apple -> edible_fruit -> fruit -> reproductive_structure
 teacher  teacher.n.01                educator                      teacher -> educator -> professional -> adult
computer computer.n.01                 machine                  computer -> machine -> device ->

In [4]:
# 8 selected word pairs covering synonyms, taxonomically close terms, and dissimilar terms
pairs = [
    ('car', 'automobile'),
    ('dog', 'cat'),
    ('dog', 'animal'),
    ('teacher', 'student'),
    ('apple', 'fruit'),
    ('chair', 'furniture'),
    ('dog', 'computer'),
    ('bank', 'cloud')
]

similarity_records = []

for w1, w2 in pairs:
    s1 = wn.synsets(w1, pos=wn.NOUN)[0]
    s2 = wn.synsets(w2, pos=wn.NOUN)[0]

    # Calculate path similarity
    sim_score = s1.path_similarity(s2)

    similarity_records.append({
        'Word 1': w1,
        'Synset 1': s1.name(),
        'Word 2': w2,
        'Synset 2': s2.name(),
        'Path Similarity': round(sim_score, 4) if sim_score else 0.0
    })

df_q3 = pd.DataFrame(similarity_records)
# Rank by similarity descending
df_q3 = df_q3.sort_values(by='Path Similarity', ascending=False).reset_index(drop=True)
print(df_q3.to_string(index=False))

 Word 1     Synset 1     Word 2       Synset 2  Path Similarity
    car     car.n.01 automobile       car.n.01           1.0000
    dog     dog.n.01     animal    animal.n.01           0.3333
  chair   chair.n.01  furniture furniture.n.01           0.3333
  apple   apple.n.01      fruit     fruit.n.01           0.3333
    dog     dog.n.01        cat       cat.n.01           0.2000
teacher teacher.n.01    student   student.n.01           0.1429
   bank    bank.n.01      cloud     cloud.n.01           0.1000
    dog     dog.n.01   computer  computer.n.01           0.0909


In [5]:
homonym_data = [
    {
        "Word": "bank",
        "Sentence": "He withdrew cash from the bank to buy groceries.",
        "Target Synset": "bank.n.02 (Financial Institution)",
        "Context Clues": "withdrew, cash, buy"
    },
    {
        "Word": "bank",
        "Sentence": "We sat on the muddy river bank and watched the sunset.",
        "Target Synset": "bank.n.01 (Sloping Land/Riverbank)",
        "Context Clues": "muddy, river, sat"
    },
    {
        "Word": "bat",
        "Sentence": "The wooden bat cracked when he hit a home run.",
        "Target Synset": "bat.n.05 (Sports Implement)",
        "Context Clues": "wooden, cracked, hit, home run"
    },
    {
        "Word": "bat",
        "Sentence": "A small brown bat flew out of the dark cave.",
        "Target Synset": "bat.n.01 (Mammal)",
        "Context Clues": "brown, flew, cave"
    },
    {
        "Word": "bark",
        "Sentence": "The rough tree bark was covered in moss.",
        "Target Synset": "bark.n.01 (Outer Tree Covering)",
        "Context Clues": "rough, tree, moss"
    },
    {
        "Word": "bark",
        "Sentence": "The guard dog will bark at any intruder.",
        "Target Synset": "bark.v.01 (Canine Sound)",
        "Context Clues": "guard, dog, intruder"
    },
    {
        "Word": "light",
        "Sentence": "Please turn off the overhead light before leaving.",
        "Target Synset": "light.n.02 (Illumination Device)",
        "Context Clues": "turn off, overhead"
    },
    {
        "Word": "light",
        "Sentence": "This backpack is extremely light and easy to carry.",
        "Target Synset": "light.a.01 (Low Weight)",
        "Context Clues": "backpack, easy to carry"
    },
    {
        "Word": "match",
        "Sentence": "He struck a match to light the campfire.",
        "Target Synset": "match.n.01 (Igniter/Friction match)",
        "Context Clues": "struck, light, campfire"
    },
    {
        "Word": "match",
        "Sentence": "The championship tennis match lasted three hours.",
        "Target Synset": "match.n.02 (Formal Contest/Game)",
        "Context Clues": "championship, tennis, lasted"
    }
]

df_q4 = pd.DataFrame(homonym_data)
print(df_q4.to_string(index=False))

 Word                                               Sentence                       Target Synset                  Context Clues
 bank       He withdrew cash from the bank to buy groceries.   bank.n.02 (Financial Institution)            withdrew, cash, buy
 bank We sat on the muddy river bank and watched the sunset.  bank.n.01 (Sloping Land/Riverbank)              muddy, river, sat
  bat         The wooden bat cracked when he hit a home run.         bat.n.05 (Sports Implement) wooden, cracked, hit, home run
  bat           A small brown bat flew out of the dark cave.                   bat.n.01 (Mammal)              brown, flew, cave
 bark               The rough tree bark was covered in moss.     bark.n.01 (Outer Tree Covering)              rough, tree, moss
 bark               The guard dog will bark at any intruder.            bark.v.01 (Canine Sound)           guard, dog, intruder
light     Please turn off the overhead light before leaving.    light.n.02 (Illumination Device)        